560. 和为 K 的子数组

给你一个整数数组 nums 和一个整数 k ，请你统计并返回 该数组中和为 k 的子数组的个数 。

子数组是数组中元素的连续非空序列。 

示例 1：

输入：nums = [1,1,1], k = 2

输出：2

示例 2：

输入：nums = [1,2,3], k = 3

输出：2
 
提示：


1 <= nums.length <= 2 * 10^4

-1000 <= nums[i] <= 1000
-10^7 <= k <= 10^7

## 解题思路

破局的核心在于两个概念：**前缀和（Prefix Sum）** 和 **初中代数（等式变换）**。我们来一步步拆解：

### 1. 什么是“前缀和”？

假设我们有一个数组 `nums`，所谓的“前缀和”，就是从第 0 个数字一直加到当前数字的总和。
如果 `nums = [1, 2, 3]`：

* 第 0 位的前缀和：`1`
* 第 1 位的前缀和：`1 + 2 = 3`
* 第 2 位的前缀和：`1 + 2 + 3 = 6`

**为什么要算前缀和？**
因为前缀和有一个超能力：**任何一段连续子数组的和，都可以用两个前缀和相减得到！**
假设你想求 `nums` 中从第 `i` 项到第 `j` 项的和，公式极其简单：


$$子数组和 = \text{前缀和}[j] - \text{前缀和}[i-1]$$

### 2. 巧妙的代数变换（核心 Aha Moment！）

既然题目要求：找到和为 `k` 的子数组。
那么带入上面的公式，就是要求：


$$\text{当前的前缀和} - \text{以前的某个前缀和} = k$$

现在，我们在遍历数组，我们非常清楚“当前的前缀和”**是多少（就是代码里的 `tmp`），我们也清楚 **`k`** 是多少。我们唯一不知道的，就是前面有没有出现过合适的**“以前的前缀和”。

利用初中代数，我们把等式移一下项：


$$\text{以前的某个前缀和} = \text{当前的前缀和} - k$$


即：


$$\text{以前的某个前缀和} = tmp - k$$

**这就是整个算法的灵魂：**
我们不再去辛辛苦苦地找“哪段子数组等于 $k$”，而是每算出一个当前前缀和 `tmp`，就回头问一句：**“喂！我前面的历史记录里，有没有哪个前缀和刚好等于 $tmp - k$ 呀？”**

### 3. 为什么要用哈希表？为什么要 `dict[0] = 1`？

* **为什么要哈希表（字典）？**
如果每次都要去历史记录里数一遍，太慢了（$O(N)$）。所以我们雇一个“图书管理员”（也就是哈希表字典 `sum_dict`），把计算过的前缀和都记下来。当我们需要寻找 `tmp - k` 时，字典能在 $O(1)$ 的时间里瞬间告诉我们它出现过几次。
* **为什么要 `dict[0] = 1`？**
假设 `k = 3`。你刚加上一个数字，当前前缀和 `tmp` 刚好等于 3！
按照公式去查：你要找的前缀和是 `tmp - k = 3 - 3 = 0`。
你要去字典里查 `0` 出现过几次。但是等等，在这个数字之前，根本没有任何数字，前缀和怎么可能是 0 呢？
这就是数学上的意义：**“什么都不加，和就是 0”。** 我们强行塞入 `{0: 1}`，相当于在数组的最前面放了一个虚拟的起点，保证了当“从头开始加刚好等于 $k$”时，公式依然完美成立。


In [ ]:
import collections
class Solution:
    def subarraySum(self, nums: List[int], k: int) -> int:
        dict_sum = defaultdict(int)
        # 前缀和的基操，前缀和为 0 的出现一次（在本题中的意思是不用减前缀，求和加起来就是k）
        dict_sum[0] = 1
        sum_num = 0
        ans = 0
        for i, num in enumerate(nums):
            sum_num += num
            # sum_num - 某个前缀和 = k -> 某个前缀和 = sum_num - k
            if sum_num - k in dict_sum:
                ans += dict_sum[sum_num - k]
            dict_sum[sum_num] += 1
        return ans